# Lab 6 — Diagnose It Yourself

Five datasets, `a` through `e`. Four of them were made by exactly one of the mechanisms from
Lecture 13 Part 2 — a **mixture** of two populations, **correlated** samples, a **heavy
tail**, or a **product** instead of a sum. One is a plain, ordinary normal sample with
nothing wrong with it.

Your job: say which is which, **including which one is fine**, and name the panel that gave
each one away. The generating code is deliberately not here.

Two things to know before you start:

* Each dataset comes with a `group` covariate, a 0/1 label. For all but one of them it is
  pure noise. Splitting by it is therefore a real diagnostic and not a giveaway.
* **Row order is meaningful. Do not sort or shuffle.** Two of the six panels read the order
  of the rows, and sorting destroys the evidence they look for.


The setup below is the toolkit from Lecture 13. The only change is one optional argument on `diagnose` (`log_panel`), used in Part 1. Read it if you want to, but you do not need to modify any of it — everything you need is in `diagnose`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline

In [ ]:
def hist_with_normal(x, bins=50, ax=None, label=None):
    if ax is None:
        ax = plt.gca()
    x = np.asarray(x)
    _, edges, _ = ax.hist(x, bins=bins, density=True, alpha=0.6, label=label)
    grid = np.linspace(edges[0], edges[-1], 300)
    ax.plot(grid, stats.norm.pdf(grid, np.mean(x), np.std(x)), "k--", label="normal, same mean & std")
    ax.legend(fontsize=8)

In [ ]:
def qq_plot(x, ax=None, label=None):
    if ax is None:
        ax = plt.gca()
    x = np.sort(np.asarray(x))
    n = len(x)
    theoretical = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)
    ax.scatter(theoretical, x, s=4, label=label)

    q25, q50, q75 = np.percentile(x, [25, 50, 75])
    slope = (q75 - q25) / (stats.norm.ppf(0.75) - stats.norm.ppf(0.25))
    ax.plot(theoretical, q50 + slope * theoretical, "k--", lw=1)
    ax.set_xlabel("normal quantile")
    ax.set_ylabel("data quantile")

In [ ]:
def running_mean(x):
    x = np.asarray(x)
    return np.cumsum(x) / np.arange(1, len(x) + 1)

In [ ]:
def autocorrelation(x, max_lag):
    x = np.asarray(x) - np.mean(x)
    denominator = np.sum(x * x)
    result = list()
    for lag in range(max_lag + 1):
        result.append(np.sum(x[:len(x) - lag] * x[lag:]) / denominator)
    return result

In [ ]:
def diagnose(x, group=None, name="", log_panel=True):
    x = np.asarray(x)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(name)

    hist_with_normal(x, ax=axes[0, 0])
    axes[0, 0].set_title("histogram vs normal")

    qq_plot(x, ax=axes[0, 1])
    axes[0, 1].set_title("Q-Q plot")

    if log_panel and np.all(x > 0):
        hist_with_normal(np.log(x), ax=axes[0, 2])
        axes[0, 2].set_title("histogram of log(x)")
    elif not log_panel:
        axes[0, 2].set_title("log panel off (data already transformed)")
    else:
        axes[0, 2].set_title("log: not possible (values <= 0)")

    axes[1, 0].plot(running_mean(x))
    axes[1, 0].set_title("running mean")

    axes[1, 1].plot(autocorrelation(x, 50), marker=".")
    axes[1, 1].axhline(0, color="k", lw=0.5)
    axes[1, 1].set_title("autocorrelation")

    if group is not None:
        bins = np.histogram_bin_edges(x, bins=40)
        axes[1, 2].hist(x[group == 0], bins=bins, alpha=0.5, density=True, label="group 0")
        axes[1, 2].hist(x[group == 1], bins=bins, alpha=0.5, density=True, label="group 1")
        axes[1, 2].legend()
    axes[1, 2].set_title("split by covariate")

    plt.tight_layout()
    plt.show()

Load the five datasets. Each name `x` has a partner `x_group` holding its covariate.

In [ ]:
mystery = np.load("Lab.6.mystery.npz")
for name in ["a", "b", "c", "d", "e"]:
    print(name, mystery[name].shape)

`diagnose` takes the data, its covariate, and a title. Here is the call for `a`; the reading of it is yours.

In [ ]:
diagnose(mystery["a"], mystery["a_group"], "mystery a")

In [ ]:
# mystery b
diagnose(mystery["b"], mystery["b_group"], "mystery b")

In [ ]:
# mystery c
diagnose(mystery["c"], mystery["c_group"], "mystery c")

In [ ]:
# mystery d
diagnose(mystery["d"], mystery["d_group"], "mystery d")

In [ ]:
# mystery e
diagnose(mystery["e"], mystery["e_group"], "mystery e")

## What to hand in

Fill in the table, in this notebook. One line of reasoning per dataset is enough — what
matters is that the panel you name is the one that actually shows the problem.

| Dataset | Which assumption broke (or none) | Which panel told you | What you would do next with real data of this kind |
|---|---|---|---|
| `a` | | | |
| `b` | | | |
| `c` | | | |
| `d` | | | |
| `e` | | | |

Two warnings, from the people who did this before you:

* One of the five passes every test of **shape** — its histogram and its Q–Q plot look
  perfectly normal. If you only look at the top row of panels, you will call it clean.
* Not every non-normal histogram is a broken assumption, and exactly one dataset here is
  fine. Resist diagnosing it anyway.


## Test your diagnosis

Diagnosing a problem is only the first step. Now test whether the action you proposed actually helps. There are three short checks.

**Do not change your original diagnosis table.** These are follow-up checks.


### 1. Test a transformation

Which dataset do you think came from a multiplicative/product process?

Enter its letter below, apply a log transformation, and run the diagnostics again.

Compare the transformed result with the original, especially the **histogram** and **Q-Q plot**.

In 2–3 sentences, answer:

- Did the transformation make the data look more compatible with a normal model?
- What changed?


In [ ]:
product_name = "a"   # enter a, b, c, d, or e

if product_name not in ["a", "b", "c", "d", "e"]:
    print("Enter one dataset letter: a, b, c, d, or e.")
else:
    x = mystery[product_name]
    g = mystery[product_name + "_group"]

    if np.all(x > 0):
        x_log = np.log(x)
        diagnose(
            x_log,
            g,
            "log-transformed mystery " + product_name,
            log_panel=False
        )
    else:
        print(
            "This dataset contains non-positive values, so log(x) is not "
            "defined for every observation. Revisit your diagnosis."
        )


**What changed after the transformation?**

Your answer:


### 2. Test the possible mixture

Which dataset do you think contains a mixture of two populations?

Choose that dataset below and examine **group 0** and **group 1** separately.

Compare:

- their means and standard deviations,
- their distributions,
- whether each group looks simpler than the combined sample.

**Note:** the running-mean and autocorrelation panels are not meaningful on a single group, because dropping rows breaks the row order. Look at the histogram and Q-Q plot here.


In [ ]:
mixture_name = ""   # enter a, b, c, d, or e

if mixture_name not in ["a", "b", "c", "d", "e"]:
    print("Enter one dataset letter: a, b, c, d, or e.")
else:
    x = mystery[mixture_name]
    g = mystery[mixture_name + "_group"]

    x0 = x[g == 0]
    x1 = x[g == 1]

    print(
        "group 0:",
        "n =", len(x0),
        "mean =", np.mean(x0),
        "std =", np.std(x0)
    )

    print(
        "group 1:",
        "n =", len(x1),
        "mean =", np.mean(x1),
        "std =", np.std(x1)
    )


In [ ]:
if mixture_name in ["a", "b", "c", "d", "e"]:
    diagnose(x0, name="group 0")
    diagnose(x1, name="group 1")


**After separating the groups, does your mixture diagnosis look more convincing? Why?**

Your answer:


### 3. Diagnosis vs. treatment

Naming a problem is not the same as fixing it. For two of the datasets, test whether the treatment you would choose actually changes the answer.

We care about one number: the **mean** of the dataset, and how precisely we know it. The **standard error (SE)** is the uncertainty on that number — smaller means more precise. The helper below estimates an SE by bootstrap: resample the data many times, recompute the statistic each time, and take the spread.

* `block=1` resamples single rows (fine when rows are independent).
* `block=50` resamples chunks of 50 consecutive rows, so neighbouring rows stay together and any dependence between them is kept.

In [ ]:
def bootstrap_se(x, stat=np.mean, block=1, n_boot=2000, seed=0):
    """Bootstrap standard error of stat(x). block > 1 resamples chunks of consecutive rows."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    n = len(x)
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n_blocks, (n_boot, n_blocks)) * block
    idx = (starts[:, :, None] + np.arange(block)).reshape(n_boot, -1)[:, :n]
    samples = x[np.minimum(idx, n - 1)]
    return np.std(stat(samples, axis=1))

**3a. The correlated dataset.** Enter its letter. Compare the naive SE (which assumes independent rows) with the block-bootstrap SE (which does not).

In [ ]:
corr_name = ""   # enter a, b, c, d, or e

if corr_name not in ["a", "b", "c", "d", "e"]:
    print("Enter one dataset letter: a, b, c, d, or e.")
else:
    x = mystery[corr_name]
    print("mean:                          ", np.mean(x))
    print("naive SE (assumes independent):", np.std(x, ddof=1) / np.sqrt(len(x)))
    print("block-bootstrap SE (block=50): ", bootstrap_se(x, np.mean, block=50))

**Your answer (2 sentences):** Which SE would you report, and what does the gap between the two tell you about how much data you really have?

Your answer:


**3b. The heavy-tailed dataset.** Enter its letter. Compare the mean with the median, and how precisely each one is estimated.

In [ ]:
tail_name = ""   # enter a, b, c, d, or e

if tail_name not in ["a", "b", "c", "d", "e"]:
    print("Enter one dataset letter: a, b, c, d, or e.")
else:
    x = mystery[tail_name]
    print("mean:   ", np.mean(x),   "  SE of mean:  ", bootstrap_se(x, np.mean))
    print("median: ", np.median(x), "  SE of median:", bootstrap_se(x, np.median))
    print("largest |value|:", np.max(np.abs(x)), "  typical spread (IQR):", np.subtract(*np.percentile(x, [75, 25])))

**Your answer (2 sentences):** Which summary would you report, and why? What would you also do about the extreme observations before trusting either number?

Your answer:


**3c. Reflection (2–3 sentences).** One of the five datasets needed no treatment at all. Which one, and what could go wrong if you "treated" it anyway (for example by logging it or switching to the median)?

Your answer:
